# Train Semi V2 — RETFound DR Grading

Pipeline này dùng trực tiếp `checkpoint-best.pth` của `ai/grading`, phát pseudo-label cho ảnh ngoài, replay dữ liệu grade có nhãn và lưu đầy đủ artifact lên Drive.

Thứ tự khi chạy mới: **Cell 1 → 6 → Train new**. Khi Colab bị ngắt: chạy lại **Cell 1 → 6 → Resume**; không chạy Train new.

In [ ]:
# CELL 1 — GPU diagnostics
import torch, sys, platform
from datetime import datetime, timezone
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('=== RUNTIME DIAGNOSTICS ===')
print('UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/dr_semi_v2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

In [ ]:
# CELL 3 — Source code và dependencies
import os, subprocess
GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/keras-grade-semi-supervised'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
print('Source ready:', REPO_DIR)

In [ ]:
# CELL 4 — Cấu hình duy nhất cho train mới, resume và test
GRADE_CHECKPOINT = Path('/content/drive/MyDrive/retfound_merged_seed42/checkpoint-best.pth')
LABELED_DATASET_DIR = Path('/content/drive/MyDrive/fundus_merged')
UNLABELED_DIR = Path('/content/drive/MyDrive/fundus_unlabeled')
RUN_NAME = 'retfound_semi_v2_seed42_threshold095'
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME
PSEUDO_CACHE_DIR = DRIVE_ROOT / 'pseudo-label-cache'

SEMI_CONFIG = {
    'epochs': 8,
    'patience': 4,
    'batch_size': 2,
    'accum_steps': 8,
    'head_lr': 1e-5,
    'backbone_lr': 1e-6,
    'min_lr': 1e-7,
    'weight_decay': 0.05,
    'threshold': 0.95,
    'pseudo_weight': 0.25,
    'num_workers': 2,
    'seed': 42,
}

for label, path in [('grade checkpoint', GRADE_CHECKPOINT), ('labeled dataset', LABELED_DATASET_DIR), ('unlabeled pool', UNLABELED_DIR)]:
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy {label}: {path}')
print('Grade checkpoint:', GRADE_CHECKPOINT)
print('Labeled replay:', LABELED_DATASET_DIR)
print('Unlabeled pool:', UNLABELED_DIR)
print('Output:', OUTPUT_DIR)
print('Reusable pseudo cache:', PSEUDO_CACHE_DIR)
print('Config:', SEMI_CONFIG)

In [ ]:
# CELL 5 — Kiểm tra contract của checkpoint grade
import json
checkpoint_metadata = torch.load(GRADE_CHECKPOINT, map_location='cpu', weights_only=False)
grade_args = checkpoint_metadata.get('args', {})
required = ['model_source', 'image_size', 'preprocessing', 'loss']
missing = [key for key in required if key not in grade_args]
if missing:
    raise ValueError(f'Checkpoint grade thiếu metadata bắt buộc: {missing}')
print(json.dumps({
    'epoch': checkpoint_metadata.get('epoch'),
    'best_qwk': checkpoint_metadata.get('best_qwk'),
    'model_source': grade_args.get('model_source'),
    'architecture': grade_args.get('architecture'),
    'model_name': grade_args.get('model_name'),
    'image_size': grade_args.get('image_size'),
    'preprocessing': grade_args.get('preprocessing'),
    'loss': grade_args.get('loss'),
    'grading_contract': checkpoint_metadata.get('grading_contract', {}),
}, indent=2, ensure_ascii=False, default=str))
del checkpoint_metadata

In [ ]:
# CELL 6 — Command builder và live logger
import shlex, signal

def build_command(*, resume=None, eval_only=False):
    c = SEMI_CONFIG
    command = [
        sys.executable, '-u', '-m', 'ai.train_semi_v2.train',
        '--checkpoint', str(GRADE_CHECKPOINT),
        '--dataset-dir', str(LABELED_DATASET_DIR),
        '--unlabeled-dir', str(UNLABELED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--pseudo-cache-dir', str(PSEUDO_CACHE_DIR),
        '--epochs', str(c['epochs']), '--patience', str(c['patience']),
        '--batch-size', str(c['batch_size']), '--accum-steps', str(c['accum_steps']),
        '--head-lr', str(c['head_lr']), '--backbone-lr', str(c['backbone_lr']),
        '--min-lr', str(c['min_lr']), '--weight-decay', str(c['weight_decay']),
        '--threshold', str(c['threshold']), '--pseudo-weight', str(c['pseudo_weight']),
        '--num-workers', str(c['num_workers']), '--seed', str(c['seed']),
    ]
    if resume is not None:
        command.extend(['--resume', str(resume)])
    if eval_only:
        command.append('--eval-only')
    return command

def run_streaming(command, log_name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log_path = OUTPUT_DIR / log_name
    print('COMMAND:', shlex.join(command))
    print('LIVE LOG:', log_path)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log_path.open('a', encoding='utf-8') as handle:
            for line in process.stdout:
                timestamped = f'[{datetime.now(timezone.utc).isoformat()}] {line}'
                print(timestamped, end='', flush=True)
                handle.write(timestamped)
                handle.flush()
        process.wait()
    except KeyboardInterrupt:
        print('\nĐang dừng an toàn. checkpoint-last.pth của epoch hoàn tất gần nhất vẫn được giữ.')
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait()
    if process.returncode not in (0, 130, -2):
        raise RuntimeError(f'Process thất bại với exit code {process.returncode}')
    return process.returncode

## Train mới

Chỉ chạy cell này khi `OUTPUT_DIR` chưa có run. Nếu cache có cùng teacher, tập ảnh, preprocessing và threshold thì pseudo-label inference được bỏ qua.

In [ ]:
# TRAIN NEW
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f'Output đã có dữ liệu: {OUTPUT_DIR}. Hãy resume hoặc đổi RUN_NAME.')
run_streaming(build_command(), 'notebook-train-live.log')

## Resume sau khi ngắt

Chạy cell này thay cho Train new. Model, optimizer, scheduler, scaler, epoch và patience được khôi phục; pseudo-label không dự đoán lại.

In [ ]:
# RESUME SEMI V2
LAST_CHECKPOINT = OUTPUT_DIR / 'checkpoint-last.pth'
if not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint resume: {LAST_CHECKPOINT}')
run_streaming(build_command(resume=LAST_CHECKPOINT), 'notebook-resume-live.log')

## Đánh giá held-out test

Chỉ chạy thủ công sau khi đã chọn checkpoint. Test không tham gia train, early stopping hoặc chọn model.

In [ ]:
# TEST CHECKPOINT
TEST_CHECKPOINT_KIND = 'best'  # 'best' hoặc 'last'
TEST_CHECKPOINT = OUTPUT_DIR / f'checkpoint-{TEST_CHECKPOINT_KIND}.pth'
if not TEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy: {TEST_CHECKPOINT}')
run_streaming(build_command(resume=TEST_CHECKPOINT, eval_only=True), 'notebook-test-live.log')
for artifact in ['test_metrics.json', 'test_predictions.csv', 'summary.json']:
    path = OUTPUT_DIR / artifact
    if path.is_file():
        print(f'\n=== {artifact} ===')
        print(path.read_text(encoding='utf-8')[:5000] if path.suffix != '.csv' else path)

## Quản lý cache

- Giữ nguyên threshold và dữ liệu: cache được dùng lại.
- Đổi threshold: tự tạo cache khác và dự đoán lại.
- Muốn buộc dự đoán lại: xóa đúng cặp `.csv` và `.json` trong `PSEUDO_CACHE_DIR`.

In [ ]:
# CACHE INVENTORY — chỉ đọc, không xóa
cache_files = sorted(PSEUDO_CACHE_DIR.glob('pseudo-labels-*')) if PSEUDO_CACHE_DIR.exists() else []
print(f'Cache directory: {PSEUDO_CACHE_DIR}')
for path in cache_files:
    print(f'  {path.name}: {path.stat().st_size / 1024**2:.2f} MiB')
if not cache_files:
    print('  (chưa có cache)')